# Ballet Pose + AI Poetry Demo (English Version)
> Input a ballet video → Real-time pose analysis + Beautiful English poetry
> No Chinese = Zero garbled text, works everywhere!

In [1]:
import os, cv2, numpy as np, torch, torch.nn as nn, random, warnings
from sklearn.cluster import KMeans
import mediapipe as mp
warnings.filterwarnings('ignore')

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
pose = mp_pose.Pose(static_image_mode=False, model_complexity=2,
                    smooth_landmarks=True, min_detection_confidence=0.5,
                    min_tracking_confidence=0.5)

In [2]:
# Body parts clustering setup
body_parts = {
    "head": list(range(0,11)),
    "left_arm": [11,13,15,17,19,21],
    "right_arm": [12,14,16,18,20,22],
    "left_leg": [23,25,27,29,31],
    "right_leg": [24,26,28,30,32],
    "torso": [11,12,23,24]
}
part_names = ["Head", "L.Arm", "R.Arm", "L.Leg", "R.Leg", "Torso"]
movement_levels = ["Still", "Slight", "Medium", "Fast", "Spin", "High"]

In [3]:
# Beautiful English poetry engine (template-based, never garbled)
POETIC_PHRASES = {
    "Spin":  ["whirls like a storm", "becomes the wind itself", "spins into pure light", "turns eternity"],
    "High":  ["reaches for the stars", "defies gravity", "soars to the heavens", "touches the sky"],
    "Fast":  ["cuts through the air", "dances with lightning", "moves like fire", "blazes across the stage"],
    "Still": ["freezes time", "becomes a sculpture of light", "holds the silence", "breathes eternity"],
    "Slight":["whispers to the wind", "barely moves yet moves everything", "trembles with grace"],
    "Medium":["flows like water", "glides through dreams", "paints the air with motion"]
}

TEMPLATES = [
    "She {move} - and the universe holds its breath.",
    "When she {move}, even the stars forget to shine.",
    "{move} - a single moment of pure freedom.",
    "In that {move}, she becomes poetry itself.",
    "She {move}, writing light with her body.",
    "Like a dream taking flight, she {move}."
]

def generate_poem_lstm(actions):
    priority = {"Spin":5, "High":4, "Fast":3, "Medium":2, "Slight":1, "Still":0}
    best_action = max(actions, key=lambda x: priority[x.split(':')[1]])
    move = best_action.split(':')[1]
    
    phrase = random.choice(POETIC_PHRASES.get(move, ["moves with grace"]))
    poem = random.choice(TEMPLATES).format(move=phrase)
    return poem.capitalize()

In [4]:
def process_and_cluster_video(video_path, track_name):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")
    
    os.makedirs(track_name, exist_ok=True)
    all_frames_data = []
    frame_count = valid_frames = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb)
        
        if results.pose_world_landmarks:
            lm = results.pose_world_landmarks.landmark
            clusters = []
            for part, indices in body_parts.items():
                coords = [lm[i].x for i in indices if i < len(lm)]
                if not coords:
                    clusters.append(0); continue
                X = np.array(coords).reshape(1,-1)
                k = KMeans(n_clusters=min(6, max(1,len(X))), random_state=42, n_init=1)
                k.fit(X)
                clusters.append(int(k.labels_[0]) % 6)
            
            if frame_count % 3 == 0:
                cv2.imwrite(os.path.join(track_name, f"frame_{valid_frames+1:06d}.jpg"), frame)
                valid_frames += 1
            all_frames_data.append(clusters)
        
        frame_count += 1
        if frame_count % 100 == 0:
            print(f"Processed {frame_count} frames → {len(all_frames_data)} valid poses")
    
    cap.release()
    print(f"\nDone! {len(all_frames_data)} poses extracted and clustered.")
    return all_frames_data

In [5]:
def run_realtime_demo(video_path, track_name):
    print("Analyzing video...")
    frames_data = process_and_cluster_video(video_path, track_name)
    
    if not frames_data:
        print("No dancer detected!")
        return
    
    print("\nDemo started! Press Q to quit\n")
    
    cap = cv2.VideoCapture(video_path)
    f_idx = d_idx = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or d_idx >= len(frames_data): break
        if f_idx % 3 != 0:
            f_idx += 1; continue
            
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb)
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        
        clusters = frames_data[d_idx]
        actions = [f"{part_names[i]}:{movement_levels[c]}" for i,c in enumerate(clusters)]
        action_str = " | ".join(actions)
        poem = generate_poem_lstm(actions)
        
        cv2.putText(frame, f"Motion: {action_str[:90]}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
        cv2.putText(frame, poem, (10, 80),
                    cv2.FONT_HERSHEY_DUPLEX, 0.9, (255,200,0), 2)
        cv2.putText(frame, f"Frame {d_idx+1}/{len(frames_data)}", (10, 120),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        
        cv2.imshow('Ballet AI Poet - Press Q to quit', frame)
        if cv2.waitKey(30) & 0xFF == ord('q'):
            break
            
        f_idx += 1
        d_idx += 1
    
    cap.release()
    cv2.destroyAllWindows()
    print("Demo finished. Thank you for watching the poetry of movement.")

In [6]:
# Run the demo!
# Change 'test04.mp4' to your own ballet video
run_realtime_demo('./dance_mp4/test04.mp4', 'demo_track')

Analyzing video...
Processed 100 frames → 100 valid poses
Processed 200 frames → 200 valid poses
Processed 300 frames → 300 valid poses
Processed 400 frames → 397 valid poses
Processed 500 frames → 497 valid poses
Processed 600 frames → 596 valid poses
Processed 700 frames → 696 valid poses
Processed 800 frames → 796 valid poses
Processed 900 frames → 896 valid poses

Done! 896 poses extracted and clustered.

Demo started! Press Q to quit

Demo finished. Thank you for watching the poetry of movement.
